In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# GitHub Not-Following-Back Analysis\n",
    "\n",
    "This notebook fetches followers and following lists for a GitHub account, identifies users not following back, and exports the results."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Import Required Libraries\n",
    "\n",
    "Import the Python libraries used for HTTP requests, data handling, and environment management."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import requests\n",
    "import pandas as pd\n",
    "from datetime import datetime, timedelta, timezone\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Configure GitHub API Access\n",
    "\n",
    "Set up the authentication token and base endpoints for the GitHub API."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "GITHUB_TOKEN = os.getenv('GITHUB_TOKEN') or os.getenv('GH_TOKEN')\n",
    "if not GITHUB_TOKEN:\n",
    "    raise RuntimeError('GitHub token is required. Set GITHUB_TOKEN or GH_TOKEN.')\n",
    "\n",
    "HEADERS = {\n",
    "    'User-Agent': 'Mozilla/5.0',\n",
    "    'Authorization': f'Bearer {GITHUB_TOKEN}',\n",
    "}\n",
    "BASE_REST = 'https://api.github.com'\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Load Target GitHub Username\n",
    "\n",
    "Define the target account and validate the input for analysis."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "TARGET_USERNAME = 'PanshulVempalli'\n",
    "if not TARGET_USERNAME:\n",
    "    raise ValueError('Target GitHub username must be set.')\n",
    "\n",
    "print('Target username:', TARGET_USERNAME)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Fetch Followers and Following\n",
    "\n",
    "Retrieve the full paginated followers and following lists from the GitHub API."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def fetch_paginated_users(endpoint_url):\n",
    "    users = []\n",
    "    page = 1\n",
    "    while True:\n",
    "        url = f'{endpoint_url}?per_page=100&page={page}'\n",
    "        response = requests.get(url, headers=HEADERS, timeout=30)\n",
    "        response.raise_for_status()\n",
    "        data = response.json()\n",
    "        if not data:\n",
    "            break\n",
    "        users.extend(item['login'] for item in data if isinstance(item, dict) and 'login' in item)\n",
    "        print(f'Fetched {len(data)} items from {endpoint_url} page {page}')\n",
    "        page += 1\n",
    "    return users\n",
    "\n",
    "followers = fetch_paginated_users(f'{BASE_REST}/users/{TARGET_USERNAME}/followers')\n",
    "following = fetch_paginated_users(f'{BASE_REST}/users/{TARGET_USERNAME}/following')\n",
    "\n",
    "print('followers_count:', len(followers))\n",
    "print('following_count:', len(following))\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Identify Users Not Following Back\n",
    "\n",
    "Compare the following list against the followers list to determine users who are not following back."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "not_following_back = sorted(set(following) - set(followers))\n",
    "print('not_following_back_count:', len(not_following_back))\n",
    "\n",
    "def user_profile(username):\n",
    "    response = requests.get(f'{BASE_REST}/users/{username}', headers=HEADERS, timeout=30)\n",
    "    response.raise_for_status()\n",
    "    return response.json()\n",
    "\n",
    "profiles = [user_profile(user) for user in not_following_back]\n",
    "\n",
    "inactive_accounts = [\n",
    "    profile['login']\n",
    "    for profile in profiles\n",
    "    if profile.get('followers', 0) < 100\n",
    "    and (\n",
    "        (datetime.fromisoformat(profile['updated_at'].replace('Z', '+00:00'))\n",
    "         if profile.get('updated_at') else None) is not None\n",
    "    )\n",
    "]\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Display and Export Results\n",
    "\n",
    "Show the non-follow-back users in a DataFrame and export the list to CSV."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df = pd.DataFrame(profiles)\n",
    "df['updated_at'] = pd.to_datetime(df['updated_at'], utc=True)\n",
    "df['inactive_bucket'] = pd.cut(\n",
    "    (pd.Timestamp.utcnow() - df['updated_at']).dt.days,\n",
    "    bins=[-1, 14, 28, 180, 365, float('inf')],\n",
    "    labels=['<14 days', '14-28 days', '28-180 days', '180-365 days', '>365 days']\n",
    "    right=True\n",
    ")\n",
    "df['below_100_followers'] = df['followers'] < 100\n",
    "result = df[(df['below_100_followers']) & (df['inactive_bucket'].isin(['180-365 days', '>365 days']))]\n",
    "print('Count matching criteria:', len(result))\n",
    "result.to_csv('not_following_back_inactive_under_100.csv', index=False)\n",
    "result.head(20)\n"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
